In [ ]:
%%writefile  custom_chunking_ingestion_pipeline_caching.py 
#can add title extraction , summary extraction
#core data structures
from llama_index.core import SimpleDirectoryReader
from llama_index.core import Document
from llama_index.core import Settings

#text_splitter
from llama_index.core.node_parser import SentenceSplitter
#embedding models
from llama_index.embeddings.openai import OpenAIEmbedding
#index
from llama_index.core import VectorStoreIndex
#LLMs config
from llama_index.llms.openai import OpenAI

from llama_index.core.ingestion import IngestionPipeline
from llama_index.core.extractors import TitleExtractor, SummaryExtractor
from llama_index.core.storage.docstore import SimpleDocumentStore 
import chromadb
from llama_index_vector_stores.chroma import ChromaVectorStore



import os 
from dotenv import load_dotenv 
load_dotenv()

Settings.llm=OpenAI(model="gpt-4o-mini")
Settings.embed_model=OpenAIEmbedding(model="text-embedding-3-small")
Settings.chunk_size=512
Settings.chunk_overlap=50
os.environ['OPENAI_API_KEY']=os.getenv("OPENAI_API_KEY")
PERSISTENCE_DIR="./pipeline_storage"
CHROMA_DIR="./chroma_db"



def main():
    documents=SimpleDirectoryReader(
    input_dir="llamaindex-docs",
    required_exts=[".md"],
    recursive=False,
    num_files_limit=10
    ).load_data()

    print(f"Loaded docs{len(documents)}")
    
    #create ingestion pipeline
    pipeline=IngestionPipeline(
        transformations=[
            SentenceSplitter(chunk_size=Settings.chunk_size,
            chunk_overlap=Settings.chunk_overlap),
            TitleExtractor(), #<= add here 
            SummaryExtractor(),
            OpenAIEmbedding(model="text-embedding-3-small")
        ],
        docstore=SimpleDocumentStore()
        )

    if os.path.exists(PERSISTENCE_DIR):
            print("Loading persisted doc store")
            pipeline.docstore.load(persist_dir=PERSISTENCE_DIR)
            print("Loaded!")

    processed_nodes=pipeline.run(documents=documents, show_progress=True)
    print(f"processed into {len(processed_nodes)}")

    pipeline.docstore.persist(persist_dir=PERSISTANCE_DIR)
    print("Persisted!")

    first_node_metadata=processed_nodes[0].metadata
    
    print("First Node metadata")
    for key, value in first_node_metadata.items():
        print(f"{key}:{value}")

    if processed_nodes[0].embedding:
        print(f"Embedding dim: {len(processed_nodes[0].embedding)}")

    #create index
    index=VectorStoreIndex(processed_nodes)
    query_engine=index.as_query_engine()
    
    query="what is llamacloud?"
    response=query_engine.query(query)

    print(response)


if __name__=="__main__":
    main()




Overwriting custom_chunking_ingestion_pipeline_caching.py


In [11]:
%%writefile  custom_chunking_ingestion_pipeline_caching_chromadb.py 
#can add title extraction , summary extraction
#core data structures
from llama_index.core import SimpleDirectoryReader
from llama_index.core import Document
from llama_index.core import Settings

#text_splitter
from llama_index.core.node_parser import SentenceSplitter
#embedding models
from llama_index.embeddings.openai import OpenAIEmbedding
#index
from llama_index.core import VectorStoreIndex
#LLMs config
from llama_index.llms.openai import OpenAI

from llama_index.core.ingestion import IngestionPipeline
from llama_index.core.extractors import TitleExtractor, SummaryExtractor
from llama_index.core.storage.docstore import SimpleDocumentStore 
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore


import os 
from dotenv import load_dotenv 
load_dotenv()

Settings.llm=OpenAI(model="gpt-4o-mini")
Settings.embed_model=OpenAIEmbedding(model="text-embedding-3-small")
Settings.chunk_size=512
Settings.chunk_overlap=50
os.environ['OPENAI_API_KEY']=os.getenv("OPENAI_API_KEY")
PERSISTENCE_DIR="./pipeline_storage"
CHROMA_DIR="./chroma_db"


def main():
    documents=SimpleDirectoryReader(
    input_dir="llamaindex-docs",
    required_exts=[".md"],
    recursive=False,
    num_files_limit=10
    ).load_data()

    print(f"Loaded docs{len(documents)}")

    chroma_client=chromadb.PersistentClient(path=CHROMA_DIR)
    chroma_collection=chroma_client.get_or_create_collection("llamaindex_docs")
    vector_store=ChromaVectorStore(chroma_collection=chroma_collection)
    
    #count docs in chroma db
    existing_count=chroma_collection.count()
    print(f"ChromaDB contains {existing_count} embeddings")

    if existing_count >0:
        print(f"using existing embeddings")
    else:
    #create ingestion pipeline
        pipeline=IngestionPipeline(
            transformations=[
                SentenceSplitter(chunk_size=Settings.chunk_size,
                chunk_overlap=Settings.chunk_overlap),
                #TitleExtractor(), #<= add here 
                #SummaryExtractor(),
                OpenAIEmbedding(model="text-embedding-3-small")
            ],
            vector_store=vector_store
            )

        processed_nodes=pipeline.run(documents=documents, show_progress=True)
        print(f"processed into {len(processed_nodes)}")

        #pipeline.docstore.persist(persist_dir=PERSISTANCE_DIR)
        #print("Persisted!")
        if processed_nodes[0].embedding:
           print(f"Embedding dim: {len(processed_nodes[0].embedding)}")

    #create index
    index=VectorStoreIndex.from_vector_store(vector_store)
    
    print("vectorstore created!")
    query_engine=index.as_query_engine()
    
    query="what is llamacloud?"
    response=query_engine.query(query)

    print(response)


if __name__=="__main__":
    main()




Overwriting custom_chunking_ingestion_pipeline_caching_chromadb.py


In [17]:
%%writefile  custom_chunking_ingestion_pipeline_caching_v1.py 
#can add title extraction , summary extraction
#core data structures
from llama_index.core import SimpleDirectoryReader
from llama_index.core import Document
from llama_index.core import Settings

#text_splitter
from llama_index.core.node_parser import SentenceSplitter
#embedding models
from llama_index.embeddings.openai import OpenAIEmbedding
#index
from llama_index.core import VectorStoreIndex
#LLMs config
from llama_index.llms.openai import OpenAI

from llama_index.core.ingestion import IngestionPipeline
from llama_index.core.extractors import TitleExtractor, SummaryExtractor
from llama_index.core.storage.docstore import SimpleDocumentStore 
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
import os 
from dotenv import load_dotenv
load_dotenv()
CHROMA_DIR="./chroma_db_cached"
CACHE_DIR="./pipeline_cache"

Settings.llm=OpenAI(model="gpt-4o-mini")
Settings.embed_model=OpenAIEmbedding(model="text-embedding-3-small")
Settings.chunk_size=512
Settings.chunk_overlap=50
os.environ['OPENAI_API_KEY']=os.getenv("OPENAI_API_KEY")


def get_transformations():
    return [SentenceSplitter(
        chunk_size=Settings.chunk_size,
        chunk_overlap=Settings.chunk_overlap
    ),
    TitleExtractor(),
    OpenAIEmbedding(model="text-embedding-3-small")
    ]

def main():
    
    documents=SimpleDirectoryReader(
    input_dir="llamaindex-docs",
    required_exts=[".md"],
    recursive=False,
    num_files_limit=5
    ).load_data()

    chroma_client=chromadb.PersistentClient(path=CHROMA_DIR)
    chroma_collection=chroma_client.get_or_create_collection("llamaindex_docs")
    vector_store=ChromaVectorStore(chroma_collection=chroma_collection)

    print(f"existing embeddings {chroma_collection.count()}")

    #create pipeline
    pipeline=IngestionPipeline(
        transformations=get_transformations(),
        vector_store=vector_store,
        docstore=SimpleDocumentStore() #track doc hashes
    )

    #load existing cache if available
    if os.path.exists(CACHE_DIR):
        print(f"Loading existing cache from {CACHE_DIR}")
        pipeline.load(persist_dir=CACHE_DIR)
        print("Cache loaded")
    else:
        print("No cache")
    print("cache transformations will be reused")
    processed_nodes=pipeline.run(documents=documents, show_progress=True)
    if processed_nodes:
        if processed_nodes[0].embedding:
            print(f"length of embedding {len(processed_nodes[0].embedding)}")
    pipeline.persist(persist_dir=CACHE_DIR)
    print("cache saved, next run will skip unchanged documents")

    vector_index=VectorStoreIndex.from_vector_store(vector_store)
    query_engine=vector_index.as_query_engine()

    query="what is llamaindex used for?"
    response=query_engine.query(query)
    print(response)

if __name__=="__main__":
    main()













Overwriting custom_chunking_ingestion_pipeline_caching_v1.py


In [5]:
!python -m pip install chromadb llama-index-vector-stores-chroma

  Using cached chromadb-1.5.0-cp39-abi3-win_amd64.whl.metadata (7.3 kB)
  Using cached uvicorn-0.40.0-py3-none-any.whl.metadata (6.7 kB)
  Using cached opentelemetry_api-1.39.1-py3-none-any.whl.metadata (1.5 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached rich-14.3.2-py3-none-any.whl.metadata (18 kB)
  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached referencing-0.37.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached rpds_py-0.30.0-cp312-cp312-win_amd64.whl.metadata (4.2 kB)
  Using cached websocket_client-1.9.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached importlib_metadata-8.7.1-py3-none-any.whl.metadata (4.7 kB)
  Using cached markdown_it_py-4.0.0-py3-none-any.whl.metadata (7.3 kB)
  Using cached huggingface_hub-1.4.1-py3-none-any.whl.metadata (13 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress t

In [13]:
!python -m custom_chunking_ingestion_pipeline_caching_chromadb

Loaded docs10
ChromaDB contains 26 embeddings
using existing embeddings
vectorstore created!
LlamaCloud is a hosted service designed for document processing and search, utilizing LlamaIndex technology. It includes several key components such as parsing complex documents into structured data, extracting well-typed data with customizable schemas, indexing document collections for searchability, classifying documents automatically, and offering features like document segmentation and intelligent table extraction.


2026-02-16 14:11:34,742 - INFO - Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
2026-02-16 14:11:35,638 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-02-16 14:11:37,070 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


In [20]:
!python -m custom_chunking_ingestion_pipeline_caching_v1

existing embeddings 11
Loading existing cache from ./pipeline_cache
Cache loaded
cache transformations will be reused
length of embedding 1536
cache saved, next run will skip unchanged documents
LlamaIndex is used for document processing and search capabilities. It helps transform complex documents into structured data, enables the extraction of well-typed data, creates searchable knowledge bases, categorizes documents automatically, and supports various advanced features like segmenting PDFs and extracting tables from spreadsheets.


2026-02-16 14:31:28,790 - INFO - Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.

Applying transformations: 100%|██████████| 3/3 [00:00<00:00, 2998.79it/s]
2026-02-16 14:31:29,490 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-02-16 14:31:30,743 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
